# Center-Pivot Field Animation — Planet PSScene

Produces an animated **GIF** and **MP4** from the Planet `analytic_sr` timeseries,
showing the alfalfa pivot during the growing season

Each frame displays:
- **Left panel** — True-colour RGB (Red/Green/Blue bands, percentile-stretched)
- **Right panel** — NDVI (RdYlGn colourmap)
- Management zone boundaries overlaid on both panels
- Date stamp and mean NDVI value
- UDM2-flagged pixels hatched in grey

Where multiple Planet strips cover the same date, the one with the most clear
pixels inside the pivot is used.

In [1]:
stationid = 'US-UTE'

## 1 · Imports

In [2]:
import io
import json
import re
import warnings
from datetime import datetime
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from matplotlib.collections import LineCollection
from PIL import Image, ImageDraw, ImageFont
import imageio.v3 as iio
import rasterio
import rasterio.features
from pyproj import Transformer
from IPython.display import Video, Image as IPImage, display

warnings.filterwarnings('ignore')
matplotlib.use('Agg')   # off-screen rendering for frame generation
print('Imports OK')

Imports OK


## 2 · Configuration

In [3]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR    = Path(f"C:/Users/dmenuz/Downloads/{stationid}")                          # directory with *_composite.tif files
GEOJSON     = Path(f'c:/Users/dmenuz/Downloads/{stationid}_aoi.geojson')          # multi-zone pivot boundary (WGS84)
OUTPUT_DIR  = Path('.')                          # where to save GIF / MP4

GIF_PATH    = OUTPUT_DIR / f'{stationid}_pivot_timeseries.gif'
MP4_PATH    = OUTPUT_DIR / f'{stationid}_pivot_timeseries.mp4'

# ── Planet band layout (1-based rasterio index) ────────────────────────────────
BAND_BLUE   = 1
BAND_GREEN  = 2
BAND_RED    = 3
BAND_NIR    = 4

SR_SCALE    = 1e-4      # analytic_sr DN → reflectance (divide by 10 000)

# ── UDM2 ──────────────────────────────────────────────────────────────────────
UDM2_CLEAR      = 1     # band index; value 1 = clear
UDM2_CONFIDENCE = 7     # 0–100
CONFIDENCE_MIN  = 70

# ── RGB percentile stretch (computed globally across all scenes below) ─────────
# Override here if you want fixed values, e.g.:
#   RGB_STRETCH = {'r': (458, 2810), 'g': (599, 2217), 'b': (384, 1586)}
RGB_STRETCH = None      # None = compute automatically

# ── NDVI display range ────────────────────────────────────────────────────────
NDVI_VMIN, NDVI_VMAX = -0.05, 0.95

# ── Zoom: pixel margin around the pivot boundary ──────────────────────────────
ZOOM_MARGIN_PX = 30     # extra pixels around the pivot bounding box

# ── Animation settings ────────────────────────────────────────────────────────
GIF_FPS     = 2         # frames per second for GIF
MP4_FPS     = 4         # frames per second for MP4
FRAME_DPI   = 120       # dots per inch for each rendered frame
FRAME_W_IN  = 11        # figure width  in inches
FRAME_H_IN  = 5.5       # figure height in inches

# ── Zone colour palette ───────────────────────────────────────────────────────
ZONE_COLORS = {'NW':'#60a5fa','NE':'#4ade80','SW':'#fb923c','SES':'#f87171','ESE':'#c084fc'}
DEFAULT_COLOR = '#e2e8f0'

print('Configuration OK')

Configuration OK


## 3 · Helper Functions

In [4]:
def ndvi(red, nir):
    """NDVI array; NaN where nodata or denom==0."""
    denom = red + nir
    with np.errstate(invalid='ignore', divide='ignore'):
        out = np.where(denom > 0, (nir - red) / denom, np.nan)
    out = np.where((red == 0) & (nir == 0), np.nan, out)
    return out.astype(np.float32)


def linear_stretch(arr, lo, hi):
    """Clip and stretch array to 0–255 uint8."""
    arr = np.clip(arr.astype(np.float32), lo, hi)
    arr = (arr - lo) / (hi - lo) * 255
    return arr.astype(np.uint8)


def collect_scenes(data_dir):
    """Return sorted list of {key, date, tif, udm2} dicts."""
    pat = re.compile(r'(\d{4}-\d{2}-\d{2})_strip_(\d+)_composite\.tif$')
    out = []
    for p in sorted(Path(data_dir).glob('*_composite.tif')):
        m = pat.match(p.name)
        if not m:
            continue
        d, sid = m.groups()
        udm2 = p.with_name(p.name.replace('_composite.tif', '_composite_udm2.tif'))
        if not udm2.exists():
            continue
        out.append({'key': f'{d}_{sid}', 'date': datetime.strptime(d, '%Y-%m-%d'),
                    'tif': p, 'udm2': udm2})
    return out


def load_scene_bands(info, all_mask):
    """Read all needed bands; return dict of arrays and quality mask."""
    with rasterio.open(info['tif']) as src:
        blue = src.read(BAND_BLUE).astype(np.float32)
        green= src.read(BAND_GREEN).astype(np.float32)
        red  = src.read(BAND_RED).astype(np.float32)
        nir  = src.read(BAND_NIR).astype(np.float32)
    with rasterio.open(info['udm2']) as usrc:
        clear= usrc.read(UDM2_CLEAR)
        conf = usrc.read(UDM2_CONFIDENCE)
    good = (clear == 1) & (conf >= CONFIDENCE_MIN)
    valid_in_pivot = int((all_mask & good & (red > 0)).sum())
    return {'blue': blue, 'green': green, 'red': red, 'nir': nir,
            'good': good, 'valid_pivot_px': valid_in_pivot}

import json
import numpy as np
import rasterio
import rasterio.features
from pyproj import Transformer

def load_geojson_zones(geojson_path, ref_tif, geojson_crs='EPSG:4326'):
    """Load zones → {name: {'mask', 'poly_px', 'area_ha'}}; also return combined mask."""
    with open(geojson_path) as f:
        gj = json.load(f)
        
    with rasterio.open(ref_tif) as src:
        epsg = src.crs.to_epsg()
        shape = (src.height, src.width)
        t = src.transform

    # FIX: Use the dynamic geojson_crs variable instead of hardcoded 'EPSG:4326'
    xf = Transformer.from_crs(geojson_crs, f'EPSG:{epsg}', always_xy=True)
    inv = ~t
    
    zones = {}
    all_mask = np.zeros(shape, dtype=bool)
    
    for feat in gj['features']:
        name = str(feat['properties'].get('quad', feat['properties'].get('id')))
        ring = feat['geometry']['coordinates'][0]
        
        utm = [xf.transform(c[0], c[1]) for c in ring]
        xs, ys = np.array([c[0] for c in utm]), np.array([c[1] for c in utm])
        
        area_ha = 0.5 * abs(np.dot(xs, np.roll(ys,-1)) - np.dot(ys, np.roll(xs,-1))) / 1e4
        poly_px = [inv * (x, y) for x, y in utm]
        
        geom = {'type': 'Polygon', 'coordinates': [utm]}
        mask = rasterio.features.geometry_mask([geom], out_shape=shape, transform=t, invert=True)
        
        all_mask |= mask
        zones[name] = {'mask': mask, 'poly_px': poly_px, 'area_ha': area_ha}
        
    return zones, all_mask, shape, t


def fig_to_pil(fig):
    """Render a matplotlib figure to a PIL Image."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=FRAME_DPI, bbox_inches='tight')
    buf.seek(0)
    return Image.open(buf).copy()


print('Functions defined')

Functions defined


## 4 · Load Zones & Compute RGB Stretch

In [5]:
# import rasterio
# with rasterio.open(ref_tif) as src:
#     print("TIF CRS:", src.crs)

In [10]:
scenes = collect_scenes(DATA_DIR)
print(f'{len(scenes)} scenes found  ({scenes[0]["date"].date()} → {scenes[-1]["date"].date()})')

ref_tif = scenes[0]['tif']
zones, all_mask, raster_shape, raster_transform = load_geojson_zones(GEOJSON, ref_tif, 'EPSG:26912')
print(f'{len(zones)} management zones loaded. Pivot area: {sum(z["area_ha"] for z in zones.values()):.1f} ha')

# Compute zoom bounding box in pixel space
rows_p, cols_p = np.where(all_mask)
M = ZOOM_MARGIN_PX
zoom = {
    'r0': max(0, rows_p.min() - M),
    'r1': min(raster_shape[0], rows_p.max() + M),
    'c0': max(0, cols_p.min() - M),
    'c1': min(raster_shape[1], cols_p.max() + M),
}
print(f'Zoom window: {zoom}')

# ── Global RGB percentile stretch ──────────────────────────────────────────────
if RGB_STRETCH is None:
    print('Computing global RGB stretch (sampling every 7th scene)…')
    rs, gs, bs = [], [], []
    for sc in scenes[::7]:
        with rasterio.open(sc['tif']) as src:
            r = src.read(BAND_RED).astype(np.float32).ravel()
            g = src.read(BAND_GREEN).astype(np.float32).ravel()
            b = src.read(BAND_BLUE).astype(np.float32).ravel()
        valid = (r > 0)
        rs.extend(r[valid][::8].tolist())
        gs.extend(g[valid][::8].tolist())
        bs.extend(b[valid][::8].tolist())
    RGB_STRETCH = {
        'r': tuple(np.percentile(rs, [2, 98])),
        'g': tuple(np.percentile(gs, [2, 98])),
        'b': tuple(np.percentile(bs, [2, 98])),
    }

print('RGB stretch:')
for ch, (lo, hi) in RGB_STRETCH.items():
    print(f'  {ch.upper()}: {lo:.0f} – {hi:.0f}')

414 scenes found  (2023-03-17 → 2025-12-03)
1 management zones loaded. Pivot area: 163.3 ha
Zoom window: {'r0': 0, 'r1': 406, 'c0': 0, 'c1': 454}
Computing global RGB stretch (sampling every 7th scene)…
RGB stretch:
  R: 549 – 2885
  G: 658 – 2285
  B: 435 – 1647


## 5 · Select Best Scene Per Date

In [11]:
print('Scoring scenes by valid pivot pixels…')

date_candidates = defaultdict(list)
for sc in scenes:
    try:
        bands = load_scene_bands(sc, all_mask)
        date_candidates[sc['date'].date()].append((bands['valid_pivot_px'], sc, bands))
    except Exception as e:
        print(f'  [skip] {sc["key"]}: {e}')

# Keep the best (most valid pixels) per date; sort chronologically
best_frames = []
for date in sorted(date_candidates):
    candidates = date_candidates[date]
    _, sc, bands = max(candidates, key=lambda x: x[0])
    best_frames.append({'date': date, 'scene': sc, 'bands': bands})

print(f'\n{len(best_frames)} unique dates → {len(best_frames)} animation frames')
print(f'Date range: {best_frames[0]["date"]} → {best_frames[-1]["date"]}')

Scoring scenes by valid pivot pixels…

333 unique dates → 333 animation frames
Date range: 2023-03-17 → 2025-12-03


## 6 · Render Frames

In [12]:
NDVI_CMAP = plt.get_cmap('RdYlGn')
NDVI_NORM = mcolors.Normalize(vmin=NDVI_VMIN, vmax=NDVI_VMAX)

def draw_zone_outlines(ax, zones, color_map, lw=1.8, ls='--', alpha=0.85):
    """Overlay zone polygon outlines (pixel coordinates)."""
    for name, z in zones.items():
        col = color_map.get(name, DEFAULT_COLOR)
        arr = np.array(z['poly_px'])     # (col, row)
        patch = mpatches.Polygon(
            arr, closed=True, fill=False,
            edgecolor=col, linewidth=lw, linestyle=ls, alpha=alpha,
        )
        ax.add_patch(patch)
        # Zone label at centroid
        cx, cy = arr[:, 0].mean(), arr[:, 1].mean()
        ax.text(cx, cy, name, color='white', fontsize=7.5, fontweight='bold',
                ha='center', va='center', zorder=5,
                bbox=dict(boxstyle='round,pad=0.15', fc=col, alpha=0.75, lw=0))


def render_frame(frame_info, stretch, ndvi_cmap, ndvi_norm, zoom, zones):
    """Build one side-by-side RGB | NDVI matplotlib figure; return PIL Image."""
    date  = frame_info['date']
    bands = frame_info['bands']
    r0, r1, c0, c1 = zoom['r0'], zoom['r1'], zoom['c0'], zoom['c1']

    # ── RGB composite ─────────────────────────────────────────────────────────
    R = linear_stretch(bands['red'],   *stretch['r'])
    G = linear_stretch(bands['green'], *stretch['g'])
    B = linear_stretch(bands['blue'],  *stretch['b'])
    rgb = np.dstack([R, G, B])[r0:r1, c0:c1]

    # Grey-out UDM2-flagged pixels
    bad_mask = ~bands['good'][r0:r1, c0:c1]
    rgb[bad_mask] = [120, 120, 120]

    # ── NDVI ──────────────────────────────────────────────────────────────────
    ndvi_arr = ndvi(bands['red'], bands['nir'])[r0:r1, c0:c1]
    ndvi_rgb  = (ndvi_cmap(ndvi_norm(ndvi_arr))[:, :, :3] * 255).astype(np.uint8)
    ndvi_rgb[bad_mask] = [80, 80, 80]          # dark grey for cloud/shadow
    nodata = np.isnan(ndvi_arr)
    ndvi_rgb[nodata] = [30, 30, 30]             # near-black for no-data

    # Mean NDVI inside pivot (good pixels only)
    pivot_good = all_mask[r0:r1, c0:c1] & bands['good'][r0:r1, c0:c1]
    mean_ndvi  = float(np.nanmean(ndvi_arr[pivot_good])) if pivot_good.any() else np.nan

    # ── Figure ────────────────────────────────────────────────────────────────
    fig, (ax_rgb, ax_ndvi) = plt.subplots(1, 2, figsize=(FRAME_W_IN, FRAME_H_IN))
    fig.patch.set_facecolor('#0f172a')

    for ax in (ax_rgb, ax_ndvi):
        ax.set_facecolor('#0f172a')
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

    ax_rgb.imshow(rgb, origin='upper', aspect='equal')
    ax_rgb.set_title('True Colour (R/G/B)', color='white', fontsize=10, pad=4)

    ax_ndvi.imshow(ndvi_rgb, origin='upper', aspect='equal')
    ax_ndvi.set_title('NDVI', color='white', fontsize=10, pad=4)

    # Colourbar for NDVI
    sm = plt.cm.ScalarMappable(cmap=ndvi_cmap, norm=ndvi_norm)
    cbar = fig.colorbar(sm, ax=ax_ndvi, fraction=0.046, pad=0.01, shrink=0.85)
    cbar.ax.tick_params(colors='white', labelsize=7)
    cbar.set_label('NDVI', color='white', fontsize=8)
    cbar.outline.set_edgecolor('#334155')

    # Zone outlines on both panels (offset to zoomed pixel coords)
    # Shift zone polygon coords to the zoomed coordinate system
    zones_zoomed = {
        name: {
            **z,
            'poly_px': [(px - c0, py - r0) for px, py in z['poly_px']],
        }
        for name, z in zones.items()
    }
    draw_zone_outlines(ax_rgb,  zones_zoomed, ZONE_COLORS)
    draw_zone_outlines(ax_ndvi, zones_zoomed, ZONE_COLORS)

    # Set consistent axis limits
    h, w = r1 - r0, c1 - c0
    for ax in (ax_rgb, ax_ndvi):
        ax.set_xlim(0, w)
        ax.set_ylim(h, 0)

    # ── Date stamp ────────────────────────────────────────────────────────────
    date_str  = date.strftime('%d %b %Y')
    ndvi_str  = f'Pivot NDVI: {mean_ndvi:.3f}' if not np.isnan(mean_ndvi) else ''
    stamp_txt = f'{date_str}    {ndvi_str}'

    fig.text(
        0.012, 0.97, stamp_txt,
        transform=fig.transFigure,
        color='white', fontsize=12.5, fontweight='bold',
        va='top', ha='left',
        path_effects=[pe.withStroke(linewidth=2.5, foreground='#0f172a')],
    )

    # Scale bar: 100 m at 3 m / pixel
    px_100m = int(100 / 3)
    bar_x0  = 0.03;  bar_y = 0.04
    bar_x1  = bar_x0 + px_100m / w * 0.46   # fraction of left panel width
    for y_off in [0, 0.005]:
        fig.add_artist(plt.Line2D(
            [bar_x0, bar_x1], [bar_y + y_off, bar_y + y_off],
            transform=fig.transFigure, color='white', lw=2.5, solid_capstyle='butt'
        ))
    fig.text(bar_x0, bar_y + 0.022, '100 m', transform=fig.transFigure,
             color='white', fontsize=7.5, va='bottom', ha='left')

    # ── Data source watermark ─────────────────────────────────────────────────
    fig.text(0.99, 0.01, 'Planet PlanetScope analytic_sr',
             transform=fig.transFigure, color='#64748b', fontsize=7,
             va='bottom', ha='right')

    plt.subplots_adjust(left=0.01, right=0.93, top=0.92, bottom=0.01, wspace=0.05)
    pil_img = fig_to_pil(fig)
    plt.close(fig)
    return pil_img


# ── Render all frames ─────────────────────────────────────────────────────────
pil_frames = []
print(f'Rendering {len(best_frames)} frames…')
for i, frame_info in enumerate(best_frames):
    img = render_frame(frame_info, RGB_STRETCH, NDVI_CMAP, NDVI_NORM, zoom, zones)
    pil_frames.append(img)
    if (i + 1) % 10 == 0 or i == len(best_frames) - 1:
        print(f'  {i+1}/{len(best_frames)}  {frame_info["date"]}')

print(f'\nAll frames rendered. Frame size: {pil_frames[0].size[0]}×{pil_frames[0].size[1]} px')

Rendering 333 frames…
  10/333  2023-04-09
  20/333  2023-04-26
  30/333  2023-05-28
  40/333  2023-06-27
  50/333  2023-07-08
  60/333  2023-07-28
  70/333  2023-08-30
  80/333  2023-09-26
  90/333  2023-10-23
  100/333  2023-11-07
  110/333  2023-12-04
  120/333  2024-04-16
  130/333  2024-05-18
  140/333  2024-06-04
  150/333  2024-06-20
  160/333  2024-07-17
  170/333  2024-08-05
  180/333  2024-09-07
  190/333  2024-10-01
  200/333  2024-10-13
  210/333  2024-11-07
  220/333  2024-11-23
  230/333  2025-04-07
  240/333  2025-04-24
  250/333  2025-05-28
  260/333  2025-06-18
  270/333  2025-07-24
  280/333  2025-08-06
  290/333  2025-08-20
  300/333  2025-09-16
  310/333  2025-10-06
  320/333  2025-10-31
  330/333  2025-11-28
  333/333  2025-12-03

All frames rendered. Frame size: 1317×657 px


## 7 · Preview First, Middle & Last Frame

In [13]:
# Switch back to inline for display
import matplotlib
matplotlib.use('Agg')

preview_indices = [0, len(pil_frames) // 2, len(pil_frames) - 1]
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
fig.patch.set_facecolor('#0f172a')
fig.suptitle('Frame Preview (first / middle / last)', color='white', fontsize=12)

for ax, idx in zip(axes, preview_indices):
    ax.imshow(pil_frames[idx])
    ax.set_title(best_frames[idx]['date'].strftime('%d %b %Y'),
                 color='white', fontsize=9)
    ax.axis('off')

plt.tight_layout()
preview_path = OUTPUT_DIR / 'animation_preview.png'
fig.savefig(preview_path, dpi=120, bbox_inches='tight', facecolor='#0f172a')
plt.show()
print(f'Preview saved → {preview_path}')

Preview saved → animation_preview.png


## 8 · Export GIF

In [ ]:
print(f'Writing GIF at {GIF_FPS} fps -> {GIF_PATH}')

# Convert frames to RGB PIL images with consistent size
pil_rgb = [f.convert('RGB') for f in pil_frames]
min_w = min(f.width  for f in pil_rgb)
min_h = min(f.height for f in pil_rgb)
pil_rgb = [f.crop((0, 0, min_w, min_h)) for f in pil_rgb]

# Quantise to 256 colours for compact GIF
pil_pal = [f.quantize(colors=256, method=Image.Quantize.MEDIANCUT) for f in pil_rgb]

pil_pal[0].save(
    GIF_PATH,
    save_all=True,
    append_images=pil_pal[1:],
    duration=int(1000 / GIF_FPS),   # ms per frame
    loop=0,                          # loop forever
    optimize=False,
)

gif_size_mb = GIF_PATH.stat().st_size / 1e6
print(f'GIF saved: {gif_size_mb:.1f} MB  ({len(pil_pal)} frames @ {GIF_FPS} fps)')
print(f'Playback duration: {len(pil_pal)/GIF_FPS:.0f} s')


## 9 · Export MP4

In [14]:
import subprocess

print(f'Writing MP4 at {MP4_FPS} fps -> {MP4_PATH}')

# Convert PIL frames to numpy RGB arrays
frames_np = [np.array(f.convert('RGB')) for f in pil_frames]

# MP4 requires even pixel dimensions
h, w = frames_np[0].shape[:2]
h_even = h - (h % 2)
w_even = w - (w % 2)
frames_np = [f[:h_even, :w_even] for f in frames_np]

# Concatenate all raw frame bytes, then pipe to ffmpeg in one call
raw_bytes = b''.join(frame.tobytes() for frame in frames_np)

ffmpeg_cmd = [
    "C:\\Users\\dmenuz\\Documents\\ffmpeg\\bin\\ffmpeg.exe", '-y',
    '-f', 'rawvideo',
    '-vcodec', 'rawvideo',
    '-s', f'{w_even}x{h_even}',
    '-pix_fmt', 'rgb24',
    '-r', str(MP4_FPS),
    '-i', 'pipe:0',
    '-vcodec', 'libx264',
    '-crf', '18',
    '-pix_fmt', 'yuv420p',
    str(MP4_PATH),
]

result = subprocess.run(ffmpeg_cmd, input=raw_bytes, capture_output=True)

if result.returncode != 0:
    print('ffmpeg error:')
    print(result.stderr.decode()[-800:])
else:
    mp4_size_mb = MP4_PATH.stat().st_size / 1e6
    print(f'MP4 saved: {mp4_size_mb:.1f} MB  ({len(frames_np)} frames @ {MP4_FPS} fps)')
    print(f'Playback duration: {len(frames_np)/MP4_FPS:.0f} s')


Writing MP4 at 4 fps -> US-UTE_pivot_timeseries.mp4
MP4 saved: 34.0 MB  (333 frames @ 4 fps)
Playback duration: 83 s


## 10 · Display in Notebook

In [ ]:
# Display GIF inline
print('GIF preview (may animate in JupyterLab):')
display(IPImage(filename=str(GIF_PATH)))

In [ ]:
# Display MP4 inline
print('MP4 preview:')
display(Video(str(MP4_PATH), embed=True, width=900))

## 11 · Summary

In [ ]:
import pandas as pd

rows = []
for frame_info in best_frames:
    bands    = frame_info['bands']
    r0,r1,c0,c1 = zoom['r0'],zoom['r1'],zoom['c0'],zoom['c1']
    ndvi_arr = ndvi(bands['red'], bands['nir'])
    pivot_good = all_mask & bands['good'] & (bands['red'] > 0)
    vals = ndvi_arr[pivot_good]
    rows.append({
        'date'         : frame_info['date'],
        'scene'        : frame_info['scene']['key'],
        'valid_px'     : int(pivot_good.sum()),
        'ndvi_mean'    : float(np.nanmean(vals)) if len(vals) else np.nan,
        'ndvi_median'  : float(np.nanmedian(vals)) if len(vals) else np.nan,
        'ndvi_max'     : float(np.nanmax(vals)) if len(vals) else np.nan,
    })

df_anim = pd.DataFrame(rows).set_index('date')
df_anim['ndvi_mean']   = df_anim['ndvi_mean'].round(3)
df_anim['ndvi_median'] = df_anim['ndvi_median'].round(3)
df_anim['ndvi_max']    = df_anim['ndvi_max'].round(3)

print('Outputs created:')
for p in [GIF_PATH, MP4_PATH, preview_path]:
    print(f'  {p.name:<35}  {p.stat().st_size/1e6:.1f} MB')

print()
print('Per-frame NDVI summary:')
df_anim[['ndvi_mean','ndvi_median','ndvi_max','valid_px']].round(3)